# Week 13 — Voice Pipeline Fundamentals
_STT, TTS, VAD, and the End-of-Utterance Decision_

---

This notebook is **standalone** — no agent, no orchestrator, no Supabase.
We're going to take apart the four mechanical pieces that turn raw
microphone audio into a spoken response, and play with the knobs that
make them feel natural.

## What you'll build understanding of

| # | Component | Question it answers |
|---|-----------|---------------------|
| 1 | **STT** (Speech-to-Text) | *What did the user say?* |
| 2 | **TTS** (Text-to-Speech) | *How do I speak this back?* |
| 3 | **VAD** (Voice Activity Detection) | *Is sound speech right now?* |
| 4 | **EOU** (End of Utterance) | *Has the user finished their thought?* |

**EOU is the most underrated concept in voice AI.** It's not the same
thing as VAD. We'll spend a whole section pulling them apart.

## Stack used here

- **Deepgram Nova-3** — STT (~300 ms streaming latency)
- **ElevenLabs Turbo v2.5** — TTS (~250 ms time-to-first-byte, premium voices)
- **Silero VAD** (ONNX) — voice activity detector
- **LiveKit Agents** — the framework that wires all four pieces together
  in production (we only *talk* about it here — actual integration is
  in Notebook 04)

> Notebook 03 also briefly compares Deepgram Aura 2 with ElevenLabs so
> you can hear the difference. Production default is ElevenLabs (configurable
> in `config/param.yaml` → `voice.tts_provider`).

---

## Section 0 — Setup

In [ ]:
# Install (or top up) every package this notebook uses, into the kernel
# that's actually running. `%pip` is the kernel-aware magic — `!pip` may
# install somewhere else if you launched Jupyter from a different Python.
# Re-running this cell is a no-op once the deps are present.
%pip install --quiet \
    python-dotenv \
    deepgram-sdk>=6.0.0 \
    elevenlabs>=2.0.0 \
    sounddevice>=0.5.0 \
    ipywidgets>=8.1.1 \
    scipy>=1.11.0 \
    matplotlib>=3.7.0 \
    numpy>=1.26.0 \
    onnxruntime>=1.17.0
print('Notebook 03 deps OK ✓')


In [ ]:
import os, sys, time, io
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

# Sanity check the one external dependency this notebook needs.
assert os.getenv("DEEPGRAM_API_KEY"), \
    "Set DEEPGRAM_API_KEY in your .env file (https://deepgram.com)"
print("DEEPGRAM_API_KEY found ✓")

In [ ]:
from deepgram import DeepgramClient

# SDK v6: the api_key MUST be a keyword argument.
dg = DeepgramClient(api_key=os.getenv("DEEPGRAM_API_KEY"))
print("Deepgram client ready (SDK v6)")

---
## Section 1 — Speech-to-Text with Deepgram Nova-3

STT is the easy part conceptually: audio bytes go in, text comes out.
What matters in production is **latency**.

Deepgram has two modes:

| Mode | API | Use case | Latency |
|------|-----|----------|---------|
| **Pre-recorded** (batch) | `transcribe_url` / `transcribe_file` | Files, async pipelines | ~1–2 seconds |
| **Streaming** (websocket) | `dg.listen.live.v("1")` | Real-time voice agents | ~300 ms |

We use the batch API in this notebook because it's simpler. The
production voice worker (Notebook 04) uses streaming via the
`livekit-plugins-deepgram` plugin — it handles the websocket for us.

In [ ]:
SAMPLE_AUDIO_URL = "https://dpgr.am/spacewalk.wav"

t0 = time.perf_counter()
response = dg.listen.v1.media.transcribe_url(
    url=SAMPLE_AUDIO_URL,
    model="nova-3",
    language="en",
    smart_format=True,   # punctuation + capitalisation
    utterances=True,     # segment by speaker turns
)
latency_ms = int((time.perf_counter() - t0) * 1000)

alt = response.results.channels[0].alternatives[0]
print(f"Latency:    {latency_ms} ms")
print(f"Confidence: {alt.confidence:.2f}")
print()
print("Transcript:")
print("  " + alt.transcript[:400])

### Why `smart_format=True` matters

Without `smart_format`, the transcript comes out as one long lower-case
blob: `"yeah as much as it's worth celebrating the first spacewalk..."`.

That's a disaster for an LLM downstream — it loses sentence boundaries,
names get lower-cased (`dr. perera` → `dr perera`), numbers stay as digits.

**Rule of thumb: always turn `smart_format` on for STT that feeds an LLM.**

---
## Section 2 — Text-to-Speech with ElevenLabs

TTS is the inverse — text in, audio out. **ElevenLabs Turbo v2.5** is
our production default: ~250 ms time-to-first-byte, the most natural
voices on the market, and configurable per-voice (the user's premium
account unlocks the full voice library).

### Choosing a voice

ElevenLabs identifies voices by ID. A few production-friendly defaults
for a hospital reception persona:

| Voice ID | Name | Style |
|----------|------|-------|
| `l7kNoIfnJKPg7779LI2t` | Aria | Plugin default — neutral female |
| `EXAVITQu4vr4xnSDxMaL` | Sarah | Warm, professional female |
| `21m00Tcm4TlvDq8ikWAM` | Rachel | Most popular ElevenLabs voice — friendly, clear |
| `pNInz6obpgDQGcFmaJgB` | Adam | Calm male |

Browse more at <https://elevenlabs.io/app/voice-library>. To swap voice
in production, edit `voice.tts_voice_id` in `config/param.yaml`.

### Why streaming matters

The ElevenLabs SDK exposes both batch (`text_to_speech.convert`) and
streaming (`text_to_speech.stream`) modes. We use **streaming** in this
notebook because the LiveKit plugin uses it too — the user starts hearing
the response while ElevenLabs is still synthesising the rest.

### Quick A/B with Deepgram Aura 2

We'll also generate one sample with Deepgram so you can hear the
difference. ElevenLabs is more expressive (better intonation, natural
breath/pause); Aura is closer to a TTS robot but ~50 ms faster TTFB.

In [ ]:
from elevenlabs import ElevenLabs
from IPython.display import Audio, display

assert os.getenv('ELEVEN_API_KEY'), \
    "Set ELEVEN_API_KEY in your .env (https://elevenlabs.io/app/settings/api-keys)"

el = ElevenLabs(api_key=os.getenv('ELEVEN_API_KEY'))
print('ElevenLabs SDK ready')

SAMPLE_TEXT = (
    "Good morning. Dr. Perera is available for cardiology consultations "
    "on Monday and Wednesday between nine AM and one PM at the Nawaloka "
    "main branch. Would you like me to book an appointment?"
)

def synth_elevenlabs(text: str, voice_id: str, out_path: str,
                     model: str = 'eleven_turbo_v2_5') -> int:
    """Stream ElevenLabs TTS to disk. Returns wall-clock latency in ms."""
    t0 = time.perf_counter()
    audio_iter = el.text_to_speech.stream(
        text=text,
        voice_id=voice_id,
        model_id=model,
        output_format='mp3_44100_128',
    )
    with open(out_path, 'wb') as f:
        for chunk in audio_iter:
            f.write(chunk)
    return int((time.perf_counter() - t0) * 1000)

# Aria — the ElevenLabs default voice (neutral female)
ms = synth_elevenlabs(SAMPLE_TEXT,
                      voice_id='l7kNoIfnJKPg7779LI2t',
                      out_path='/tmp/tts_aria.mp3')
print(f'Aria  (eleven_turbo_v2_5): {ms} ms')
display(Audio('/tmp/tts_aria.mp3'))

In [ ]:
# Try a different ElevenLabs voice — Rachel (most popular, warm professional)
ms = synth_elevenlabs(SAMPLE_TEXT,
                      voice_id='21m00Tcm4TlvDq8ikWAM',
                      out_path='/tmp/tts_rachel.mp3')
print(f'Rachel (eleven_turbo_v2_5): {ms} ms')
display(Audio('/tmp/tts_rachel.mp3'))

In [ ]:
# A/B with Deepgram Aura 2 — same text, different provider.
# Notice the slightly faster TTFB but flatter intonation.
from deepgram import DeepgramClient
dg = DeepgramClient(api_key=os.getenv('DEEPGRAM_API_KEY'))

t0 = time.perf_counter()
audio_iter = dg.speak.v1.audio.generate(
    text=SAMPLE_TEXT, model='aura-2-andromeda-en', encoding='mp3',
)
with open('/tmp/tts_aura.mp3', 'wb') as f:
    for chunk in audio_iter:
        f.write(chunk)
print(f'aura-2-andromeda-en (Deepgram): {int((time.perf_counter() - t0) * 1000)} ms')
display(Audio('/tmp/tts_aura.mp3'))

**Listen carefully.** ElevenLabs (Aria, Rachel) sounds noticeably more
human — natural breathing, sentence rhythm, expressive emphasis on names
like "Dr. Perera". Deepgram Aura 2 is competent and ~50 ms faster, but
flatter on prosody. For a hospital reception persona where warmth
matters, ElevenLabs is worth the extra cost; for a fast-paced support
line where every millisecond counts, Aura 2 is the right call.

> Switching providers is a one-line change in `config/param.yaml`:
> `voice.tts_provider: elevenlabs` ↔ `deepgram`. The voice agent
> in Notebook 04 reads this config at startup.

---
## Section 3 — Voice Activity Detection with Silero

VAD answers exactly one question, every 32 milliseconds:

> **Is sound speech right now? Yes or no.**

It doesn't transcribe. It doesn't understand meaning. It just looks at
acoustic energy and outputs a probability between 0.0 and 1.0.

We use **Silero VAD** — a ~1.6 MB ONNX model that runs on CPU in
sub-millisecond time per frame. It's the same model LiveKit's
`livekit-plugins-silero` package ships, so we'll load it directly
from there to avoid downloading anything.

In [ ]:
import numpy as np
import onnxruntime as ort
import wave

# Locate the ONNX model bundled with the LiveKit Silero plugin.
import livekit.plugins.silero as _silero
MODEL_PATH = Path(_silero.__file__).parent / "resources" / "silero_vad.onnx"
assert MODEL_PATH.exists(), f"Silero ONNX not found at {MODEL_PATH}"

vad_session = ort.InferenceSession(str(MODEL_PATH))
print(f"Loaded Silero VAD")
print(f"  inputs:  {[i.name for i in vad_session.get_inputs()]}")
print(f"  outputs: {[o.name for o in vad_session.get_outputs()]}")

In [ ]:
# We need a real speech sample — synthetic tones won't trigger VAD.
# The silero-vad repo ships a 60s test clip we can pull via torch.hub.
TEST_WAV = Path.home() / ".cache/torch/hub/snakers4_silero-vad_master/tests/data/test.wav"

if not TEST_WAV.exists():
    print("Downloading silero-vad test audio (one-time)...")
    import torch
    torch.hub.load("snakers4/silero-vad", "silero_vad", onnx=True, force_reload=False)

with wave.open(str(TEST_WAV), "rb") as wf:
    SAMPLE_RATE = wf.getframerate()
    raw = wf.readframes(wf.getnframes())
    audio = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0

audio = audio[: 10 * SAMPLE_RATE]   # first 10 seconds for a clean plot
print(f"Audio: {len(audio)/SAMPLE_RATE:.1f}s @ {SAMPLE_RATE} Hz")

In [ ]:
def run_silero(audio: np.ndarray, sample_rate: int, chunk: int = 512):
    """Run Silero VAD on ``audio`` and return per-chunk speech probabilities."""
    state = np.zeros((2, 1, 128), dtype=np.float32)   # hidden state
    probs = []
    for i in range(0, len(audio) - chunk, chunk):
        frame = audio[i : i + chunk].reshape(1, -1)
        out, state = vad_session.run(
            None,
            {
                "input": frame,
                "state": state,
                "sr": np.array(sample_rate, dtype=np.int64),
            },
        )
        probs.append(float(out[0][0]))
    return np.array(probs)

probs = run_silero(audio, SAMPLE_RATE)
print(f"Frames processed:    {len(probs)}")
print(f"Frames > 0.5 thresh: {(probs > 0.5).sum()} ({(probs > 0.5).mean()*100:.0f}%)")
print(f"Max probability:     {probs.max():.3f}")

In [ ]:
import matplotlib.pyplot as plt

CHUNK = 512
frame_times = np.arange(len(probs)) * (CHUNK / SAMPLE_RATE)
audio_times = np.arange(len(audio)) / SAMPLE_RATE

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(audio_times, audio, alpha=0.6, linewidth=0.3, color="#2c3e50")
ax1.set_ylabel("Amplitude")
ax1.set_title("Raw audio waveform (real human speech)")

ax2.plot(frame_times, probs, color="#e74c3c", linewidth=1.4)
ax2.axhline(0.5, color="gray", linestyle="--", label="threshold = 0.5")
ax2.fill_between(frame_times, probs, alpha=0.2, color="#e74c3c")
ax2.set_ylabel("P(speech)")
ax2.set_xlabel("Time (seconds)")
ax2.set_title("Silero VAD output (probability of speech per 32 ms frame)")
ax2.legend(loc="upper right")
ax2.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

Notice how the probability **spikes** during speech and **drops to near
zero** during silence. The threshold (0.5 by default) is what turns this
continuous signal into binary `speaking / not speaking`.

**Key insight:** the VAD output is a *signal*, not a *decision*.
It tells you when sound is happening. It does **not** tell you when
the user has finished talking. That second question is a different
thing entirely — and that thing is **EOU**.

---
## Section 4 — End of Utterance (EOU): the concept that makes voice feel human

If you walk away from this notebook with one idea, make it this:

> **VAD is a sensor. EOU is a decision.**

VAD gives us a stream of `is_speaking?` flags every 32 ms. EOU is the
*policy* that watches that stream and decides: *the user has finished —
commit the transcript and start thinking*.

### Why getting EOU wrong feels awful

| EOU policy | What happens |
|------------|-------------|
| **Too aggressive** (decides too early) | Agent cuts you off mid-sentence after a pause for breath. "What's my appoint—" *agent already responding* |
| **Too conservative** (waits forever) | You finish speaking, then sit in silence for 3 seconds before the agent realises you're done. Conversation feels dead |
| **Just right** | Natural turn-taking. Pauses for thought are tolerated, but the moment you sound finished, the agent moves |

### How the EOU decision is built (three layers stacked)

```
 Audio frame (32 ms)
        ↓
 ┌────────────────────────────────────────────────────────────┐
 │ Layer 1 — VAD                                              │
 │   Silero outputs P(speech) per frame.                      │
 │   threshold = 0.5  →  is_speaking? Yes/No                  │
 └────────────────────────────────────────────────────────────┘
        ↓
 ┌────────────────────────────────────────────────────────────┐
 │ Layer 2 — Silence accumulator                              │
 │   How long has VAD been saying "no" continuously?          │
 │   silence_threshold_ms = 500  →  candidate end of speech   │
 └────────────────────────────────────────────────────────────┘
        ↓
 ┌────────────────────────────────────────────────────────────┐
 │ Layer 3 — Endpointing buffer                               │
 │   Wait an extra `min_endpointing_delay` to confirm.        │
 │   If VAD pops back to "yes" during the buffer, abort.      │
 │   min_endpointing_delay = 0.5 s  →  CONFIRMED EOU          │
 └────────────────────────────────────────────────────────────┘
        ↓
 STT finalises transcript → LLM starts thinking
```

### The four knobs you tune

| Knob | What it does | Make it bigger ⇒ | Make it smaller ⇒ |
|------|--------------|------------------|-------------------|
| `vad_threshold` | Probability cutoff for "is speech" | Misses quiet/whispered speech | More false triggers from noise |
| `silence_threshold_ms` | How much continuous silence to call it "a pause" | Tolerates longer pauses | Fires on every breath |
| `min_endpointing_delay` | Confirmation buffer after silence detected | More conservative; waits longer | Faster response but may cut off |
| Deepgram STT `endpointing_ms` (server-side) | Streaming STT's own end-of-segment hint | More confident transcripts | Faster commit, more retries |

### Three sensible presets

| Preset | `vad_threshold` | `silence_threshold_ms` | `min_endpointing_delay` | Feels like |
|--------|----------------:|-----------------------:|------------------------:|-----------|
| Aggressive | 0.4 | 250 | 0.2 | Snappy. May cut you off |
| **Balanced (default)** | **0.5** | **500** | **0.5** | Natural |
| Conservative | 0.6 | 800 | 0.8 | Patient. Safe for slow / non-native speakers |

---
## Section 5 — EOU parameter playground

Let's *see* the difference. We'll re-segment the same audio under three
EOU policies and look at where each policy decides the speaker has
stopped.

Here we implement layers 1 & 2 (VAD + silence accumulator) directly so
we can plot the segment boundaries. Layer 3 (`min_endpointing_delay`)
would just shift the boundaries to the right by that many seconds.

In [ ]:
from dataclasses import dataclass

@dataclass
class EOUPolicy:
    name: str
    vad_threshold: float
    silence_ms: int            # silence_threshold_ms
    endpoint_buffer_s: float   # min_endpointing_delay


def detect_utterances(probs: np.ndarray, sample_rate: int, chunk: int,
                      policy: EOUPolicy) -> list[tuple[float, float]]:
    """Return list of (start_s, end_s) utterances given a per-frame VAD trace."""
    frame_s = chunk / sample_rate
    silence_frames_needed = int(policy.silence_ms / 1000 / frame_s)

    utterances = []
    in_speech = False
    speech_start = 0
    silence_run = 0

    for i, p in enumerate(probs):
        is_speech = p >= policy.vad_threshold
        t = i * frame_s

        if is_speech:
            if not in_speech:
                in_speech = True
                speech_start = t
            silence_run = 0
        else:
            if in_speech:
                silence_run += 1
                if silence_run >= silence_frames_needed:
                    end_t = t - policy.silence_ms / 1000 + policy.endpoint_buffer_s
                    utterances.append((speech_start, end_t))
                    in_speech = False
                    silence_run = 0
    if in_speech:
        utterances.append((speech_start, len(probs) * frame_s))
    return utterances


POLICIES = [
    EOUPolicy("Aggressive",   0.4, 250, 0.2),
    EOUPolicy("Balanced",     0.5, 500, 0.5),
    EOUPolicy("Conservative", 0.6, 800, 0.8),
]

for p in POLICIES:
    utts = detect_utterances(probs, SAMPLE_RATE, CHUNK, p)
    print(f"{p.name:15s} → {len(utts):3d} utterances")
    for s, e in utts[:5]:
        print(f"    {s:5.2f}s → {e:5.2f}s   ({e - s:.2f}s long)")
    if len(utts) > 5:
        print(f"    ... and {len(utts) - 5} more")
    print()

In [ ]:
# Visualise all three policies on the same VAD trace.
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
colors = ["#e74c3c", "#3498db", "#27ae60"]

for ax, policy, color in zip(axes, POLICIES, colors):
    ax.plot(frame_times, probs, color="#7f8c8d", linewidth=0.8, alpha=0.7)
    ax.axhline(policy.vad_threshold, color=color, linestyle="--", linewidth=1, alpha=0.6)
    ax.fill_between(frame_times, probs, alpha=0.15, color="#7f8c8d")

    utts = detect_utterances(probs, SAMPLE_RATE, CHUNK, policy)
    for s, e in utts:
        ax.axvspan(s, e, alpha=0.25, color=color)

    ax.set_ylabel("P(speech)")
    ax.set_title(f"{policy.name}  ·  VAD={policy.vad_threshold}, silence={policy.silence_ms}ms, buffer={policy.endpoint_buffer_s}s  →  {len(utts)} utterances")
    ax.set_ylim(-0.05, 1.05)

axes[-1].set_xlabel("Time (seconds)")
plt.tight_layout()
plt.show()

**What to notice:**

- Aggressive splits a single speaker into many short utterances —
  every breath becomes a turn boundary.
- Conservative merges everything into a few long utterances — natural
  for monologues but feels laggy in dialogue.
- Balanced sits in the middle. This is what we ship to production.

---
## Section 6 — Live recording widget

Now let's actually capture *your* voice. We use `sounddevice` for raw
PCM capture and `ipywidgets` for the Start / Stop buttons.

When you click **Start**, a callback streams audio into a buffer. When
you click **Stop**, the buffer becomes a numpy array — same shape as the
test audio above. You can plug it straight into the VAD code from
Section 3 if you want.

In [ ]:
import sounddevice as sd
import ipywidgets as widgets
from scipy.io import wavfile

REC_SR = 16000  # Deepgram + Silero both expect 16 kHz mono

_rec_state = {"frames": [], "running": False, "stream": None}

def _audio_cb(indata, frames, time_info, status):
    if _rec_state["running"]:
        _rec_state["frames"].append(indata.copy())

btn_start = widgets.Button(
    description="Start", icon="microphone",
    button_style="danger",
    layout=widgets.Layout(width="160px", height="40px"),
)
btn_stop = widgets.Button(
    description="Stop", icon="stop",
    button_style="warning", disabled=True,
    layout=widgets.Layout(width="160px", height="40px"),
)
status = widgets.HTML(value="<i>Click Start, then say something</i>")
playback = widgets.Output()

# These globals will be populated when the user stops recording.
recorded_audio: np.ndarray | None = None
recorded_wav_bytes: bytes | None = None

def _on_start(_):
    global recorded_audio, recorded_wav_bytes
    _rec_state["frames"] = []
    _rec_state["running"] = True
    _rec_state["stream"] = sd.InputStream(
        samplerate=REC_SR, channels=1, dtype="int16", callback=_audio_cb,
    )
    _rec_state["stream"].start()
    btn_start.disabled, btn_stop.disabled = True, False
    status.value = "<b style='color:red'>● Recording... speak now</b>"

def _on_stop(_):
    global recorded_audio, recorded_wav_bytes
    _rec_state["running"] = False
    if _rec_state["stream"] is not None:
        _rec_state["stream"].stop()
        _rec_state["stream"].close()
        _rec_state["stream"] = None
    btn_start.disabled, btn_stop.disabled = False, True

    if not _rec_state["frames"]:
        status.value = "<b style='color:orange'>No audio captured.</b>"
        return

    recorded_audio = np.concatenate(_rec_state["frames"], axis=0).flatten()
    duration = len(recorded_audio) / REC_SR
    buf = io.BytesIO()
    wavfile.write(buf, REC_SR, recorded_audio)
    recorded_wav_bytes = buf.getvalue()

    status.value = (
        f"<b style='color:green'>✓ Recorded {duration:.1f}s "
        f"({len(recorded_wav_bytes)/1024:.1f} KB)</b>"
    )
    with playback:
        playback.clear_output(wait=True)
        display(Audio(data=recorded_audio, rate=REC_SR))

btn_start.on_click(_on_start)
btn_stop.on_click(_on_stop)

display(widgets.VBox([widgets.HBox([btn_start, btn_stop]), status, playback]))

---
## Section 7 — A mini pipeline (no agent)

Now we wire just three of the four pieces together — STT → (skip LLM) → TTS —
to feel the latency budget.

**Sample queries to try** (record one of these in the widget above):

- *"Hello, my name is Anushka"*
- *"What are the visiting hours at Nawaloka Hospital?"*
- *"Can you tell me what cardiology services are offered?"*
- *"I'd like to book an appointment with a cardiologist"*

These are the same routes the agent will exercise in Notebook 04.

In [ ]:
if recorded_wav_bytes is None:
    print("⚠ No recording yet — go back to Section 6 and click Start / Stop first.")
else:
    # Step 1 — STT (Deepgram, batch)
    t0 = time.perf_counter()
    stt_resp = dg.listen.v1.media.transcribe_file(
        request=recorded_wav_bytes,
        model='nova-3', language='en', smart_format=True,
    )
    stt_ms = int((time.perf_counter() - t0) * 1000)
    transcript = stt_resp.results.channels[0].alternatives[0].transcript
    print(f"[STT  {stt_ms:4d} ms]  \"{transcript}\"")

    if not transcript.strip():
        print("  (no speech detected — try speaking louder or closer to the mic)")
    else:
        # Step 2 — pretend the LLM said something. Just echo the user's words back.
        echo = f"You said: {transcript}"

        # Step 3 — TTS (ElevenLabs Turbo v2.5, streaming)
        t0 = time.perf_counter()
        tts_ms = synth_elevenlabs(
            echo,
            voice_id='l7kNoIfnJKPg7779LI2t',  # Aria
            out_path='/tmp/echo.mp3',
        )
        print(f"[TTS  {tts_ms:4d} ms]  echo audio ready (ElevenLabs)")
        print()
        print(f"Total non-LLM latency: {stt_ms + tts_ms} ms")
        print("  (The agent's LLM call adds another 1–3 seconds on top.)")
        display(Audio('/tmp/echo.mp3'))

---
## Section 8 — LiveKit concepts (theory)

We've now seen STT, TTS, VAD, and EOU as separate pieces. The question
is: in production, what wires them together and connects to the
browser's microphone?

Answer: **LiveKit Agents**, a real-time voice framework built on WebRTC.
We don't run LiveKit in this notebook (it needs a server), but here's
the mental model.

### The four LiveKit primitives

| Primitive | What it is |
|-----------|------------|
| **Room** | A virtual space. Browsers and agent workers join the same room and exchange audio/video tracks |
| **Agent** | Your pipeline definition: `instructions + STT + LLM + TTS + VAD`. Stateless |
| **AgentSession** | A *running instance* of an Agent — bound to one Room, one participant. Stateful |
| **JobContext** | Hands you a Room reference at startup; this is where your worker code begins |

### How it plumbs together

```
 ┌──────────────────┐         ┌──────────────────┐
 │  User Browser    │  WebRTC │  LiveKit Cloud   │
 │  mic ───────────────audio──>  Room: "call-42" │
 │  speaker <───────────audio──                  │
 └──────────────────┘         └────────┬─────────┘
                                       │ (joins room)
                                       ▼
                        ┌───────────────────────────┐
                        │  Agent Worker (our code)  │
                        │  ┌─────────────────────┐  │
                        │  │ Silero VAD          │  │
                        │  │   ↓ 32 ms frames    │  │
                        │  │ Deepgram STT        │  │
                        │  │   ↓ transcript      │  │
                        │  │ Our LLM adapter     │  │
                        │  │   ↓ response text   │  │
                        │  │ Deepgram TTS        │  │
                        │  │   ↓ audio bytes     │  │
                        │  └─────────────────────┘  │
                        └───────────────────────────┘
```

### What's special about it

- **VAD runs continuously** — even *while TTS is playing*. That's how
  barge-in works: if you start talking, the agent's TTS gets cut off
  and a new turn begins.
- **STT and TTS are streaming** — Deepgram websockets, not
  request/response. Latency tracks the network, not the audio length.
- **Cloud-managed transport** — we don't run TURN/STUN servers; LiveKit
  Cloud handles WebRTC NAT traversal.

**Notebook 04 walks through the actual code** — the `LangGraphLLMAdapter`
that plugs the LLM slot of LiveKit's `Agent` into our orchestrator's
`achat()`.

---
## Summary

| Concept | What it does | Tunable |
|---------|--------------|---------|
| **STT** (Deepgram Nova-3) | Audio → text | `smart_format`, `language`, model size |
| **TTS** (ElevenLabs Turbo v2.5) | Text → audio | `tts_voice_id`, `tts_model`; switch provider via `tts_provider` |
| **VAD** (Silero ONNX) | Frame-level `is_speech?` | `vad_threshold` (0.5) |
| **EOU** (policy on top of VAD) | Decides "user is done" | `silence_threshold_ms` (500), `min_endpointing_delay` (0.5s) |
| **LiveKit Agent** | Wires it all together over WebRTC | Plugin choice, persona |

**Three things to remember:**

1. **VAD ≠ EOU.** VAD is a sensor; EOU is a decision built on top.
2. **Latency lives in EOU, not in STT.** STT itself is ~300 ms; the
   `silence_threshold_ms + min_endpointing_delay` budget often adds
   more than that.
3. **Streaming everywhere.** Batch mode is fine for notebooks but
   doubles or triples production latency. LiveKit's plugins stream by
   default — both the Deepgram and ElevenLabs ones.

**Next:** Notebook 04 plugs all of this into our Week 13 multi-agent
orchestrator and shows the full voice pipeline running end-to-end.